# Superstore Sales Analysis with SQL

Week 3 assignment. The goal here is to answer a set of business questions on the
Superstore dataset using **subqueries, CTEs and window functions**.

I'm loading the raw CSV into a small SQLite database (no server, no setup), splitting
it into `customers`, `orders` and `products`, and then running each query. Every
result also gets written to the `output/` folder as a CSV so I can check them after
running.

One thing about grain: each row in the data is a single **order line**, so `orders`
holds order lines. When a question asks about an *order value* I roll the lines up to
the order id; when it asks about a single sale I use the line's sales figure.

In [1]:
import re
import sqlite3
from pathlib import Path

import pandas as pd

base = Path.cwd()
data_file = base / "data" / "superstore.csv"
out_dir = base / "output"
out_dir.mkdir(exist_ok=True)

db_path = out_dir / "superstore.db"

try:
    conn.close()
except NameError:
    pass

conn = sqlite3.connect(db_path)


def run(sql, name=None):
    result = pd.read_sql_query(sql, conn)
    if name:
        result.to_csv(out_dir / f"{name}.csv", index=False)
    return result

In [2]:
raw = pd.read_csv(data_file)
raw.columns = [re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower() for c in raw.columns]
raw.to_sql("superstore_raw", conn, if_exists="replace", index=False)

print(raw.shape)
raw.head()

(9994, 21)


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Step 1 — Set up the tables

Three tables built straight from `superstore_raw`, filled with `SELECT DISTINCT`.
`customers` and `orders` have clean keys; `products` can carry a couple of name
variants for the same id, which is fine here since the sales analysis is
customer-based.

In [3]:
schema = '''
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;

CREATE TABLE customers (
    customer_id   TEXT PRIMARY KEY,
    customer_name TEXT,
    segment       TEXT
);

CREATE TABLE products (
    product_id   TEXT,
    category     TEXT,
    sub_category TEXT,
    product_name TEXT
);

CREATE TABLE orders (
    row_id      INTEGER PRIMARY KEY,
    order_id    TEXT,
    order_date  TEXT,
    ship_date   TEXT,
    ship_mode   TEXT,
    customer_id TEXT,
    product_id  TEXT,
    region      TEXT,
    city        TEXT,
    state       TEXT,
    postal_code TEXT,
    sales       REAL,
    quantity    INTEGER,
    discount    REAL,
    profit      REAL
);

INSERT INTO customers
SELECT DISTINCT customer_id, customer_name, segment
FROM superstore_raw;

INSERT INTO products
SELECT DISTINCT product_id, category, sub_category, product_name
FROM superstore_raw;

INSERT INTO orders
SELECT DISTINCT row_id, order_id, order_date, ship_date, ship_mode,
       customer_id, product_id, region, city, state, postal_code,
       sales, quantity, discount, profit
FROM superstore_raw;
'''

conn.executescript(schema)
conn.commit()

for t in ["customers", "products", "orders"]:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", conn)["n"][0]
    print(f"{t:<10} {n}")

customers  793
products   1894
orders     9994


## Step 2 — Required queries

**2.1 — Orders above the average sale.** Subquery in the `WHERE` clause comparing
each line against the overall average.

In [4]:
q1 = run("""
SELECT row_id, order_id, customer_id, ROUND(sales, 2) AS sales
FROM orders
WHERE sales > (SELECT AVG(sales) FROM orders)
ORDER BY sales DESC
""", "01_orders_above_average")

print(f"{len(q1)} order lines above the average sale")
q1.head(10)

2360 order lines above the average sale


,row_id,order_id,customer_id,sales
0,2698,CA-2014-145317,SM-20320,22638.48
1,6827,CA-2016-118689,TC-20980,17499.95
2,8154,CA-2017-140151,RB-19360,13999.96
3,2624,CA-2017-127180,TA-21385,11199.97
4,4191,CA-2017-166709,HL-15040,10499.97
5,9040,CA-2016-117121,AB-10105,9892.74
6,4099,CA-2014-116904,SC-20095,9449.95
7,4278,US-2016-107440,BS-11365,9099.93
8,8489,CA-2016-158841,SE-20110,8749.95
9,6426,CA-2016-143714,CC-12370,8399.98


**2.2 — Highest single sale per customer.** Correlated subquery: keep the line whose
sales equals that customer's maximum.

In [5]:
q2 = run("""
SELECT o.customer_id, c.customer_name, o.order_id, ROUND(o.sales, 2) AS sales
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.sales = (
    SELECT MAX(o2.sales) FROM orders o2 WHERE o2.customer_id = o.customer_id
)
ORDER BY o.sales DESC
""", "02_highest_sale_per_customer")

q2.head(10)

,customer_id,customer_name,order_id,sales
0,SM-20320,Sean Miller,CA-2014-145317,22638.48
1,TC-20980,Tamara Chand,CA-2016-118689,17499.95
2,RB-19360,Raymond Buch,CA-2017-140151,13999.96
3,TA-21385,Tom Ashbrook,CA-2017-127180,11199.97
4,HL-15040,Hunter Lopez,CA-2017-166709,10499.97
5,AB-10105,Adrian Barton,CA-2016-117121,9892.74
6,SC-20095,Sanjit Chand,CA-2014-116904,9449.95
7,BS-11365,Bill Shonely,US-2016-107440,9099.93
8,SE-20110,Sanjit Engle,CA-2016-158841,8749.95
9,CC-12370,Christopher Conant,CA-2016-143714,8399.98


**2.3 — Total sales per customer.** CTE that groups the lines, then joined back for
the name.

In [6]:
q3 = run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT ct.customer_id, c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY ct.total_sales DESC
""", "03_total_sales_per_customer")

q3.head(10)

,customer_id,customer_name,total_sales
0,SM-20320,Sean Miller,25043.05
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.34
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57
5,KL-16645,Ken Lonsdale,14175.23
6,SC-20095,Sanjit Chand,14142.33
7,HL-15040,Hunter Lopez,12873.30
8,SE-20110,Sanjit Engle,12209.44
9,CC-12370,Christopher Conant,12129.07


**2.4 — Customers above the average total.** Same CTE, filtered against the average
of the totals with a subquery.

In [7]:
q4 = run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT ct.customer_id, c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
ORDER BY ct.total_sales DESC
""", "04_above_average_customers")

print(f"{len(q4)} customers are above the average total")
q4.head(10)

294 customers are above the average total


,customer_id,customer_name,total_sales
0,SM-20320,Sean Miller,25043.05
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.34
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57
5,KL-16645,Ken Lonsdale,14175.23
6,SC-20095,Sanjit Chand,14142.33
7,HL-15040,Hunter Lopez,12873.30
8,SE-20110,Sanjit Engle,12209.44
9,CC-12370,Christopher Conant,12129.07


**2.5 — Rank customers by total sales.** `RANK()` over the totals.

In [8]:
q5 = run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name,
       ROUND(ct.total_sales, 2) AS total_sales,
       RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY sales_rank
""", "05_customer_rank")

q5.head(10)

,customer_name,total_sales,sales_rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5
5,Ken Lonsdale,14175.23,6
6,Sanjit Chand,14142.33,7
7,Hunter Lopez,12873.30,8
8,Sanjit Engle,12209.44,9
9,Christopher Conant,12129.07,10


**2.6 — Number each line within a customer.** `ROW_NUMBER()` partitioned by customer,
ordered by sales so line 1 is that customer's biggest.

In [9]:
q6 = run("""
SELECT customer_id, order_id, row_id, ROUND(sales, 2) AS sales,
       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY sales DESC) AS line_no
FROM orders
ORDER BY customer_id, line_no
""", "06_line_number_within_customer")

q6.head(12)

,customer_id,order_id,row_id,sales,line_no
0,AA-10315,CA-2016-103982,5199,3930.07,1
1,AA-10315,CA-2014-128055,2230,673.57,2
2,AA-10315,CA-2016-103982,5201,431.98,3
3,AA-10315,CA-2017-147039,1160,362.94,4
4,AA-10315,CA-2014-128055,2231,52.98,5
5,AA-10315,CA-2016-103982,5202,41.72,6
6,AA-10315,CA-2015-121391,1300,26.96,7
7,AA-10315,CA-2014-138100,7469,14.94,8
8,AA-10315,CA-2014-138100,7470,14.56,9
9,AA-10315,CA-2017-147039,1161,11.54,10


**2.7 — Top 3 customers by total sales.** Rank inside a CTE, then keep ranks 1-3.

In [10]:
q7 = run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked AS (
    SELECT c.customer_name, ct.total_sales,
           RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
    FROM customer_totals ct
    JOIN customers c ON c.customer_id = ct.customer_id
)
SELECT customer_name, ROUND(total_sales, 2) AS total_sales, sales_rank
FROM ranked
WHERE sales_rank <= 3
ORDER BY sales_rank
""", "07_top3_customers")

q7

,customer_name,total_sales,sales_rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3


## Step 3 — Final combined query

Customer name, total sales and rank in one shot: JOIN + CTE + window function.

In [11]:
final = run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name,
       ROUND(ct.total_sales, 2) AS total_sales,
       RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY sales_rank
""", "08_final_customer_rank")

final.head(10)

,customer_name,total_sales,sales_rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5
5,Ken Lonsdale,14175.23,6
6,Sanjit Chand,14142.33,7
7,Hunter Lopez,12873.30,8
8,Sanjit Engle,12209.44,9
9,Christopher Conant,12129.07,10


## Mini project — Customer Sales Insights

**Top 5 customers.**

In [12]:
top5 = run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders GROUP BY customer_id
)
SELECT c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY ct.total_sales DESC
LIMIT 5
""", "09_top5_customers")

top5

,customer_name,total_sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57


**Bottom 5 customers.**

In [13]:
bottom5 = run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders GROUP BY customer_id
)
SELECT c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
ORDER BY ct.total_sales ASC
LIMIT 5
""", "10_bottom5_customers")

bottom5

,customer_name,total_sales
0,Thais Sissman,4.83
1,Lela Donovan,5.30
2,Carl Jackson,16.52
3,Mitch Gastineau,16.74
4,Roy Skaria,22.33


**Customers with only one order.** Counting distinct order ids, not lines.

In [14]:
one_order = run("""
SELECT c.customer_name, COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY o.customer_id, c.customer_name
HAVING COUNT(DISTINCT o.order_id) = 1
ORDER BY c.customer_name
""", "11_single_order_customers")

print(f"{len(one_order)} customers placed exactly one order")
one_order

12 customers placed exactly one order


,customer_name,order_count
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1
5,Lela Donovan,1
6,Mitch Gastineau,1
7,Patricia Hirasaki,1
8,Ricardo Emerson,1
9,Roland Murray,1


**Customers with above-average sales** (same result as 2.4, kept here for the
mini-project checklist).

In [15]:
above_avg = run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders GROUP BY customer_id
)
SELECT c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
FROM customer_totals ct
JOIN customers c ON c.customer_id = ct.customer_id
WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
ORDER BY ct.total_sales DESC
""", "12_above_average_customers")

print(f"{len(above_avg)} customers")
above_avg.head(10)

294 customers


,customer_name,total_sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
5,Ken Lonsdale,14175.23
6,Sanjit Chand,14142.33
7,Hunter Lopez,12873.30
8,Sanjit Engle,12209.44
9,Christopher Conant,12129.07


**Highest order value per customer.** Roll the lines up to the order id first, then
take each customer's biggest order.

In [16]:
highest_order = run("""
WITH order_values AS (
    SELECT customer_id, order_id, SUM(sales) AS order_value
    FROM orders
    GROUP BY customer_id, order_id
)
SELECT c.customer_name, ROUND(MAX(ov.order_value), 2) AS highest_order_value
FROM order_values ov
JOIN customers c ON c.customer_id = ov.customer_id
GROUP BY ov.customer_id, c.customer_name
ORDER BY highest_order_value DESC
""", "13_highest_order_value")

highest_order.head(10)

,customer_name,highest_order_value
0,Sean Miller,23661.23
1,Tamara Chand,18336.74
2,Raymond Buch,14052.48
3,Tom Ashbrook,13716.46
4,Becky Martin,10539.90
5,Hunter Lopez,10499.97
6,Sanjit Chand,9900.19
7,Adrian Barton,9892.74
8,Bill Shonely,9135.19
9,Sanjit Engle,8805.04


## Notes

A few things that stood out from the results:

- Sales are very top-heavy. Sean Miller leads with about 25,043, then Tamara Chand
  (~19,052) and Raymond Buch (~15,117), while the smallest customers barely clear a
  few dollars.
- Only 294 of the 793 customers sit above the average total (~2,897). The average is
  pulled up by a handful of large accounts, so most customers land below it.
- Just 12 customers ordered a single time, so repeat buyers are the norm.
- The largest single order in the whole dataset comes to ~23,661.

All query outputs are saved under `output/` (one CSV per query) plus the SQLite file
`superstore.db`, so the results can be reopened without re-running everything.

In [17]:
conn.close()